# Sweep analysis: AMM vs CLOB vs Hybrid

Loads factorial sweep outputs and produces publication figures under `analysis/figures/`.

In [1]:
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

_cwd = Path.cwd()
if (_cwd / "results" / "sweep_summary.parquet").exists():
    ROOT = _cwd
elif (_cwd / "analysis" / "results" / "sweep_summary.parquet").exists():
    ROOT = _cwd / "analysis"
else:
    raise FileNotFoundError("Cannot locate analysis/results/sweep_summary.parquet")

RESULTS = ROOT / "results"
FIGURES = ROOT / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "legend.fontsize": 10,
    "figure.dpi": 120,
    "axes.grid": False,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

MECH_ORDER = ["amm", "clob", "hybrid"]
MECH_LABEL = {"amm": "AMM", "clob": "CLOB", "hybrid": "Hybrid"}
MECH_COLOR = {"amm": "#d4842a", "clob": "#3a5f7d", "hybrid": "#5b8c5a"}
MECH_MARKER = {"amm": "o", "clob": "s", "hybrid": "^"}
CAP_ORDER = ["low", "mid", "high"]
SIG_ORDER = ["low", "high"]
REL_BAND = 0.02
SAVE_DPI = 150

df_summary = pd.read_parquet(RESULTS / "sweep_summary.parquet")
df_ts = pd.read_parquet(RESULTS / "sweep_timeseries.parquet")

print(f"summary: {len(df_summary)} rows")
print(f"timeseries: {len(df_ts)} rows")

summary: 900 rows
timeseries: 119301 rows


In [2]:
# Fair prices per (seed, signal_regime) — static world labels for trajectory plots only.
import sys
sys.path.insert(0, str(ROOT.parent))
from simulation.sweep import SIGNAL_REGIMES, _default_information_config
from environment import InformationEnvironment

fair_records: list[dict] = []
for _, row in df_summary[["seed", "signal_regime", "signal_noise_std", "tail_noise_std"]].drop_duplicates().iterrows():
    seed = int(row["seed"])
    sig_reg = row["signal_regime"]
    cfg = _default_information_config(
        signal_noise_std=float(row["signal_noise_std"]),
        tail_noise_std=float(row["tail_noise_std"]),
    )
    _, child_world, _ = np.random.SeedSequence(seed).spawn(3)
    world_rng = np.random.default_rng(child_world)
    info_env = InformationEnvironment(cfg, world_rng)
    for m in range(info_env.n_markets):
        fair_records.append(
            {
                "seed": seed,
                "signal_regime": sig_reg,
                "market_id": m,
                "fair_price": float(info_env.world.truths[m].fair_price),
            }
        )
df_fair = pd.DataFrame(fair_records)

meta_cols = ["run_id", "mechanism", "mix", "capital_band", "signal_regime", "seed"]
df_meta = df_summary[meta_cols].drop_duplicates()

ts = df_ts.merge(df_meta, on="run_id", how="left")
ts = ts.merge(df_fair, on=["seed", "signal_regime", "market_id"], how="left")
ts["rel_err_mkt"] = (ts["mid"] - ts["fair_price"]).abs() / ts["fair_price"]

ts_run = (
    ts.groupby(["run_id", "tick", "mechanism", "mix", "capital_band", "signal_regime", "seed"], as_index=False)[
        "rel_err_mkt"
    ].max()
    .rename(columns={"rel_err_mkt": "rel_err"})
)
ts_diverse = ts_run[ts_run["mix"] == "diverse"].copy()
print(f"trajectory rows (diverse): {len(ts_diverse)}")

trajectory rows (diverse): 10574


In [3]:
# Figure 1 — Rent efficiency by mechanism (stabilized denominator)
RENT_COL = "rent_efficiency_stable"
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharey=True)
x = np.arange(len(CAP_ORDER))
width = 0.25

for ax, sig in zip(axes, SIG_ORDER):
    sub = df_summary[df_summary["signal_regime"] == sig]
    for i, mech in enumerate(MECH_ORDER):
        means, stds = [], []
        for cap in CAP_ORDER:
            vals = sub[(sub["mechanism"] == mech) & (sub["capital_band"] == cap)][RENT_COL]
            means.append(vals.mean())
            stds.append(vals.std(ddof=1) if len(vals) > 1 else 0.0)
        offset = (i - 1) * width
        ax.bar(
            x + offset,
            means,
            width,
            yerr=stds,
            capsize=3,
            label=MECH_LABEL[mech],
            color=MECH_COLOR[mech],
            edgecolor="black",
            linewidth=0.6,
            error_kw={"elinewidth": 1, "ecolor": "black"},
        )
    ax.axhline(0, color="black", linewidth=0.8, linestyle="-")
    ax.axhline(-1, color="gray", linewidth=0.8, linestyle="--")
    ax.set_xticks(x)
    ax.set_xticklabels([c.capitalize() for c in CAP_ORDER])
    ax.set_xlabel("Capital regime")
    ax.set_title(f"Signal: {sig}")

axes[0].set_ylabel("Rent efficiency (stable)")
axes[1].legend(loc="lower right", frameon=True)
fig.suptitle(
    "Rent efficiency by mechanism, capital, and signal regime\n"
    "Metric: informed PnL / max(|noise loss|, $10). Negative = informed pay rent.\n"
    "Error bars: ±1 std across 25 seeds",
    fontsize=11,
)
fig.tight_layout()
p1 = FIGURES / "01_rent_efficiency.png"
fig.savefig(p1, dpi=SAVE_DPI, bbox_inches="tight")
plt.close(fig)
print(f"Saved {p1}")

Saved /Users/adityabhosale/Downloads/Projects/orderbook-amm-hybrid-sim/analysis/figures/01_rent_efficiency.png


In [4]:
# Figure 2 — Convergence curves (diverse mix)
cap_colors = {"low": "#4c72b0", "mid": "#55a868", "high": "#c44e52"}
sig_ls = {"low": "-", "high": "--"}

fig, axes = plt.subplots(1, 3, figsize=(13, 4.2), sharey=True)

for ax, mech in zip(axes, MECH_ORDER):
    for cap in CAP_ORDER:
        for sig in SIG_ORDER:
            mask = (
                (ts_diverse["mechanism"] == mech)
                & (ts_diverse["capital_band"] == cap)
                & (ts_diverse["signal_regime"] == sig)
            )
            grp = ts_diverse.loc[mask].groupby("tick")["rel_err"]
            if grp.ngroups == 0:
                continue
            mean = grp.mean()
            std = grp.std(ddof=1)
            label = f"{cap}/{sig}"
            ax.plot(
                mean.index,
                mean.values,
                color=cap_colors[cap],
                linestyle=sig_ls[sig],
                linewidth=1.6,
                label=label,
            )
            ax.fill_between(
                mean.index,
                mean - std,
                mean + std,
                color=cap_colors[cap],
                alpha=0.12,
                linewidth=0,
            )
    ax.axhline(REL_BAND, color="black", linewidth=0.9, linestyle=":", label="2% band")
    ax.set_title(MECH_LABEL[mech])
    ax.set_xlabel("Tick")
    ax.set_xlim(left=0)

axes[0].set_ylabel("Max relative price error |mid - fair| / fair")
handles, labels = axes[-1].get_legend_handles_labels()
axes[-1].legend(handles, labels, fontsize=8, ncol=2, loc="upper right", frameon=True)
fig.suptitle(
    "Price convergence over time by mechanism (diverse mix)\n"
    "Lines: mean across seeds; shaded: ±1 std. Line style = signal; color = capital.",
    fontsize=11,
)
fig.tight_layout()
p2 = FIGURES / "02_convergence_curves.png"
fig.savefig(p2, dpi=SAVE_DPI, bbox_inches="tight")
plt.close(fig)
print(f"Saved {p2}")

Saved /Users/adityabhosale/Downloads/Projects/orderbook-amm-hybrid-sim/analysis/figures/02_convergence_curves.png


In [5]:
# Figure 3 — Capital saturation
fig, axes = plt.subplots(1, 2, figsize=(10, 4.2), sharey=True)
x = np.arange(len(CAP_ORDER))

for ax, sig in zip(axes, SIG_ORDER):
    sub = df_summary[df_summary["signal_regime"] == sig]
    for mech in MECH_ORDER:
        means, stds = [], []
        for cap in CAP_ORDER:
            vals = sub[(sub["mechanism"] == mech) & (sub["capital_band"] == cap)][
                "frac_informed_exhausted_before_convergence"
            ]
            means.append(vals.mean())
            stds.append(vals.std(ddof=1) if len(vals) > 1 else 0.0)
        ax.errorbar(
            x,
            means,
            yerr=stds,
            label=MECH_LABEL[mech],
            color=MECH_COLOR[mech],
            marker=MECH_MARKER[mech],
            markersize=7,
            linewidth=1.8,
            capsize=4,
            elinewidth=1,
        )
    ax.set_xticks(x)
    ax.set_xticklabels([c.capitalize() for c in CAP_ORDER])
    ax.set_xlabel("Capital regime")
    ax.set_title(f"Signal: {sig}")
    ax.set_ylim(bottom=-0.02)

axes[0].set_ylabel("Fraction exhausted before convergence")
axes[1].legend(loc="best", frameon=True)
fig.suptitle(
    "Fraction of informed agents exhausted before convergence\n"
    "Shaded error bars: ±1 std across 25 seeds",
    fontsize=11,
)
fig.tight_layout()
p3 = FIGURES / "03_capital_saturation.png"
fig.savefig(p3, dpi=SAVE_DPI, bbox_inches="tight")
plt.close(fig)
print(f"Saved {p3}")

Saved /Users/adityabhosale/Downloads/Projects/orderbook-amm-hybrid-sim/analysis/figures/03_capital_saturation.png


In [6]:
# Figure 4 — Sample mid-price trajectories (seed 0, diverse, mid capital, high signal)
PICK_SEED = 0
PICK_MIX = "diverse"
PICK_CAP = "mid"
PICK_SIG = "high"

ts_pick = ts[
    (ts["mix"] == PICK_MIX)
    & (ts["capital_band"] == PICK_CAP)
    & (ts["signal_regime"] == PICK_SIG)
    & (ts["seed"] == PICK_SEED)
].copy()

fair_pick = df_fair[(df_fair["seed"] == PICK_SEED) & (df_fair["signal_regime"] == PICK_SIG)]

fig, axes = plt.subplots(1, 3, figsize=(12, 4), sharex=True)
for ax, mkt in zip(axes, sorted(ts_pick["market_id"].unique())):
    fair_m = float(fair_pick.loc[fair_pick["market_id"] == mkt, "fair_price"].iloc[0])
    ax.axhline(fair_m, color="black", linestyle="--", linewidth=1.2, label="Fair value")
    for mech in MECH_ORDER:
        rid = f"{mech}__{PICK_MIX}__{PICK_CAP}__{PICK_SIG}__seed{PICK_SEED}"
        curve = ts_pick[(ts_pick["run_id"] == rid) & (ts_pick["market_id"] == mkt)].sort_values("tick")
        ax.plot(
            curve["tick"],
            curve["mid"],
            color=MECH_COLOR[mech],
            linestyle="-" if mech != "clob" else "-",
            marker=MECH_MARKER[mech],
            markevery=max(1, len(curve) // 12),
            markersize=5,
            linewidth=1.5,
            label=MECH_LABEL[mech],
        )
    ax.set_title(f"Market {mkt}")
    ax.set_xlabel("Tick")

axes[0].set_ylabel("Mid price")
axes[-1].legend(loc="best", frameon=True)
fig.suptitle(
    f"Sample mid-price trajectories vs true fair value\n"
    f"Seed {PICK_SEED}, diverse mix, {PICK_CAP} capital, {PICK_SIG} signal",
    fontsize=11,
)
fig.tight_layout()
p4 = FIGURES / "04_price_trajectories.png"
fig.savefig(p4, dpi=SAVE_DPI, bbox_inches="tight")
plt.close(fig)
print(f"Saved {p4}")

Saved /Users/adityabhosale/Downloads/Projects/orderbook-amm-hybrid-sim/analysis/figures/04_price_trajectories.png


In [7]:
# Summary table: diverse / mid / low signal (25 seeds per mechanism)
slice_df = df_summary[
    (df_summary["mix"] == "diverse")
    & (df_summary["capital_band"] == "mid")
    & (df_summary["signal_regime"] == "low")
]
headline = (
    slice_df.groupby("mechanism")
    .agg(
        convergence_error=("normalized_rmse_log", "mean"),
        rent_efficiency=("rent_efficiency", "mean"),
        rent_efficiency_stable=("rent_efficiency_stable", "mean"),
        capital_saturation=("frac_informed_exhausted_before_convergence", "mean"),
        n_trades_mean=("n_trades", "mean"),
    )
    .reindex(MECH_ORDER)
    .round(4)
)
headline.index = [MECH_LABEL[m] for m in headline.index]
from IPython.display import display

display(headline)

,convergence_error,rent_efficiency,rent_efficiency_stable,capital_saturation,n_trades_mean
AMM,0.0076,8.1570,1.3474,0.0,35.16
CLOB,0.0087,8.9636,2.1334,0.0,32.20
Hybrid,0.0084,1.4156,0.7350,0.0,17.24


## Diagnostic slices (same sweep, no rerun)

Headline reference: diverse / **mid** capital / **low** signal (table above).

In [ ]:
def _slice_table(mix: str, capital: str, signal: str) -> pd.DataFrame:
  """Mean metrics over 25 seeds for one factorial cell."""
  sl = df_summary[
      (df_summary["mix"] == mix)
      & (df_summary["capital_band"] == capital)
      & (df_summary["signal_regime"] == signal)
  ]
  out = (
      sl.groupby("mechanism")
      .agg(
          convergence=("normalized_rmse_log", "mean"),
          rent_efficiency_stable=("rent_efficiency_stable", "mean"),
          capital_saturation=("frac_informed_exhausted_before_convergence", "mean"),
          n_trades_mean=("n_trades", "mean"),
          n_trades_std=("n_trades", "std"),
      )
      .reindex(MECH_ORDER)
      .round(4)
  )
  out.index = [MECH_LABEL[m] for m in out.index]
  out["n_trades"] = out.apply(
      lambda r: f"{r['n_trades_mean']:.1f} ± {r['n_trades_std']:.1f}", axis=1
  )
  return out[
      ["convergence", "rent_efficiency_stable", "capital_saturation", "n_trades"]
  ]


print("Table A — diverse / low capital / low signal (25 seeds)")
display(_slice_table("diverse", "low", "low"))

**Table A comment:** Convergence ordering barely changes (all ~0.0084–0.0087; Hybrid slightly best). **Rent efficiency reshuffles sharply:** headline had CLOB > AMM > Hybrid (1.35–2.13), but at low capital all three collapse toward zero (0.09–0.28) with AMM on top — capital binding removes the informed edge. Trade counts drop ~15–20% vs headline but **AMM > CLOB > Hybrid** is unchanged. Capital saturation still zero.

In [ ]:
print("Table B — diverse / mid capital / high signal (25 seeds)")
display(_slice_table("diverse", "mid", "high"))

**Table B comment:** Convergence is similar across mechanisms (~0.009); CLOB is marginally best. **Rent efficiency flips sign:** headline informed “win” (positive stable rent) becomes **negative for AMM and Hybrid** under noisy signals, with only CLOB near break-even (0.07). So the headline rent ordering does **not** hold — high signal noise helps noise traders and erodes informed PnL. Trade-count ordering **AMM > CLOB > Hybrid** still holds; levels are close to headline for AMM/CLOB. Capital saturation remains zero.

## Headline findings

Across the factorial sweep (900 runs, 25 seeds per cell), **rent efficiency is negative for all three mechanisms** in most capital and signal settings: informed traders pay rent to liquidity providers and noise rather than extracting it. **Hybrid** tends to sit between **AMM** and **CLOB** on rent efficiency in the diverse/mid/low slice, while **CLOB** (with bootstrap liquidity) achieves the lowest terminal log-RMSE in that slice but similar rent drag to AMM. **Convergence** to within the 2% relative-error band is fastest under continuous AMM liquidity; CLOB and hybrid piecewise mids converge more slowly in mean trajectory, especially at low capital. **Capital saturation** (fraction of informed agents exhausting budget before convergence) remains near zero in the current parameterization because budgets are large relative to trade size and margin requirements.

### Diverse mix, mid capital, low signal (means over 25 seeds)

See the table above for the mechanism-by-metric comparison (convergence error, rent efficiency, capital saturation, mean trade count).

### Limitations and next steps

- **25 seeds per cell** is adequate for means but thin for tail quantiles or rare-event rates (exhaustion, convergence failure).
- **LP strategy is fixed** (`lp_spread_pct`, `lp_base_qty`, `lp_levels`); no sweep over passive LP depth or spread.
- **CLOB bootstrap** uses a fixed ladder (`agent_id=-1`); sensitivity to anchor spread and depth not explored.
- **Maker/taker threshold** is static; agents do not adapt posting vs taking with learning or inventory.
- **Rent and PnL metrics** use terminal fair-value mark-to-market; no path-dependent utility or fees beyond the venue stubs.
- **Three markets, diverse mix only** in convergence panels; naive-dominated mix omitted from time-series figures.
- **Next:** sweep LP and bootstrap parameters; increase seeds for high-signal tails; add maker-rebate or fee schedule; compare maker share and fill rates by mechanism.